# SentinelAI — Helmet detector, split A (chia theo video)

Train lại YOLO11s trên split chia **theo video** (không còn leakage giữa train/val/test).
Recipe giống hệt checkpoint cũ (`helmet_best.pt`), chỉ đổi cách chia dữ liệu.

- `RUN = "3cls"`: helmet, motorcycle, no-helmet — so sánh với checkpoint cũ.
- `RUN = "4cls"`: thêm plate, `fliplr=0` (không lật ngang biển số).

Lưu ý: checkpoint cũ được đánh giá trên test split bị leak, nên số mới không so sánh like-for-like.

Cách chạy: Add Data → dataset `helmet_split_a` (private). Accelerator: GPU T4. Internet: On.
Save Version → Save & Run All (Commit). Kết quả: `results_<RUN>.zip` trong tab Output.

In [ ]:
# Cell 1 — Cài đặt (khóa phiên bản để khớp lần train cũ)
!pip install -q ultralytics==8.4.26

In [ ]:
# Cell 2 — Cấu hình và dựng dataset 3 hoặc 4 class
import glob, os, json, shutil, platform
import yaml

RUN = "3cls"   # "3cls": so sánh công bằng với checkpoint cũ | "4cls": thêm plate
BASE = "yolo11s.pt"  # hoặc "yolo11n.pt"
NAME = f"helmet_{RUN}_{BASE[:7]}_splitA"   # ví dụ helmet_3cls_yolo11s_splitA
KEEP = {"3cls": ["helmet", "motorcycle", "no-helmet"],
        "4cls": ["helmet", "motorcycle", "no-helmet", "plate"]}[RUN]
SPLITS = ["train", "val", "test", "extra_1080p"]

yamls = glob.glob("/kaggle/input/**/data.yaml", recursive=True)
assert len(yamls) == 1, f"Cần đúng 1 data.yaml, thấy: {yamls}"
SRC = os.path.dirname(yamls[0])
src_names = yaml.safe_load(open(yamls[0]))["names"]
if isinstance(src_names, dict):
    src_names = [src_names[k] for k in sorted(src_names)]
missing = [n for n in KEEP if n not in src_names]
assert not missing, f"Không thấy class {missing} trong {src_names}"
remap = {src_names.index(n): i for i, n in enumerate(KEEP)}  # id gốc -> id mới
print("Class gốc:", src_names, "| giữ lại:", KEEP)

DS = "/kaggle/working/ds"
shutil.rmtree(DS, ignore_errors=True)
stats = {}
for sp in SPLITS:
    img_dir = os.path.join(SRC, "images", sp)
    if not os.path.isdir(img_dir):
        continue
    os.makedirs(f"{DS}/images/{sp}")
    os.makedirs(f"{DS}/labels/{sp}")
    n_img, counts = 0, [0] * len(KEEP)
    for img in sorted(os.listdir(img_dir)):
        stem, ext = os.path.splitext(img)
        if ext.lower() not in (".jpg", ".jpeg", ".png"):
            continue
        os.symlink(os.path.join(img_dir, img), f"{DS}/images/{sp}/{img}")  # không copy ảnh
        out = []
        lab = os.path.join(SRC, "labels", sp, stem + ".txt")
        if os.path.exists(lab):
            for line in open(lab):
                parts = line.split()
                if parts and int(parts[0]) in remap:
                    new_id = remap[int(parts[0])]
                    out.append(" ".join([str(new_id)] + parts[1:]))
                    counts[new_id] += 1
        with open(f"{DS}/labels/{sp}/{stem}.txt", "w") as f:
            f.write("\n".join(out) + ("\n" if out else ""))
        n_img += 1
    stats[sp] = {"images": n_img, "instances": dict(zip(KEEP, counts))}

names = {i: n for i, n in enumerate(KEEP)}
yaml.safe_dump({"path": DS, "train": "images/train", "val": "images/val",
                "test": "images/test", "names": names},
               open(f"{DS}/data.yaml", "w"), sort_keys=False)
print(json.dumps(stats, indent=2))

# Kiểm tra khớp SPLIT_INFO.md: train 1026 / val 214 / test 214 / extra_1080p 6 ảnh
expected = {"train": 1026, "val": 214, "test": 214, "extra_1080p": 6}
for sp, n in expected.items():
    assert stats.get(sp, {}).get("images") == n, f"{sp}: {stats.get(sp)} != {n} ảnh"
print("Số ảnh khớp SPLIT_INFO.")

In [ ]:
# Cell 3 — Train (recipe giống hệt checkpoint cũ, chỉ đổi split)
from ultralytics import YOLO

model = YOLO(BASE)
model.train(
    data=f"{DS}/data.yaml", epochs=150, patience=25, imgsz=640, batch=32, device=0,
    seed=0, deterministic=True, optimizer="auto", cos_lr=True,
    mosaic=1.0, mixup=0.1, degrees=10.0, translate=0.1, scale=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5,   # cho cả 3cls và 4cls: lật ngang không hại cho việc PHÁT HIỆN biển số; chỉ OCR mới cần chữ đúng chiều, và OCR là bước riêng chạy trên crop
    project="/kaggle/working/runs", name=NAME, exist_ok=True,
    # KHÔNG đặt time=: khi có time, Ultralytics tính lại số epoch từ thời gian mỗi epoch nên lịch cosine khác (EXP-016: ≈178 epoch thay vì 150),
    # làm kết quả không so sánh được với EXP-015. (Lần chạy 4cls đầu tiên bị treo hơn 3.9 giờ với output 0 B: xem EXP-016.)
    workers=2,   # mặc định 8: nghi ngờ nghẽn nạp dữ liệu trên Kaggle (chưa xác minh)
)

In [ ]:
# Cell 4 — Đánh giá theo class trên val, test và extra_1080p
import torch, ultralytics

best = f"/kaggle/working/runs/{NAME}/weights/best.pt"
m = YOLO(best)
report = {
    "run": RUN, "split": "A (grouped by video)", "classes": KEEP, "splits": stats,
    "env": {"ultralytics": ultralytics.__version__, "torch": torch.__version__,
            "gpu": torch.cuda.get_device_name(0), "python": platform.python_version()},
    "eval": {},
}
for sp in ["val", "test", "extra_1080p"]:
    if sp not in stats:
        continue
    y = f"{DS}/eval_{sp}.yaml"
    yaml.safe_dump({"path": DS, "train": "images/train", "val": f"images/{sp}",
                    "names": names}, open(y, "w"), sort_keys=False)
    r = m.val(data=y, split="val", project="/kaggle/working/runs",
              name=f"eval_{RUN}_{sp}", exist_ok=True, plots=True)
    per = {}
    for j, c in enumerate(r.box.ap_class_index):
        p, rec, ap50, ap = r.box.class_result(j)
        per[KEEP[int(c)]] = {"P": round(float(p), 4), "R": round(float(rec), 4),
                             "mAP50": round(float(ap50), 4), "mAP50-95": round(float(ap), 4)}
    report["eval"][sp] = {
        "all": {"P": round(float(r.box.mp), 4), "R": round(float(r.box.mr), 4),
                "mAP50": round(float(r.box.map50), 4), "mAP50-95": round(float(r.box.map), 4)},
        "per_class": per,
    }

print(json.dumps(report["eval"], indent=2))

In [ ]:
# Cell 5 — Đóng gói kết quả để tải về
json.dump(report, open(f"/kaggle/working/runs/metrics_{RUN}.json", "w"), indent=2)
shutil.rmtree(DS, ignore_errors=True)  # xóa symlink để output gọn
shutil.make_archive(f"/kaggle/working/results_{RUN}", "zip", "/kaggle/working/runs")
print(f"Xong. Tải file results_{RUN}.zip ở tab Output.")